### Installation

In [2]:
# %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# !pip install transformers==4.57.1
# !pip install --no-deps trl==0.22.2

### Setting variables

In [ ]:
# Pretrained model name
PRETRAINED_MODEL = "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit"

# Path to finetunig data
DATA_DIR = "/scratch/rsvargh2/coco/occluded_images/noise"

# Path to save model checkpoints
MODEL_SAVE_PATH = "/scratch/rsvargh2/finetuned_huggingface_models/unsloth/coco/Llama-3.2-11B-Vision-Instruct-sft-lora/"

### Unsloth

In [2]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# Unsloth models can be found at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    PRETRAINED_MODEL,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/etc/python/sitecustomize.py:117: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Mllama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A30. Num GPUs = 1. Max memory: 23.598 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.98s/it]


We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [3]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = False, # False if not finetuning MLP layers

    r = 2,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 4,  # Recommended alpha == r at least
    lora_dropout = 0.25,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    target_modules = "all-linear"  # ["qkv", "v_proj"] => List of modules, "all-linear" => Linear module
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.25.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model.vision_model.transformer` require gradients


<a name="Data"></a>
### Data Prep


In [4]:
import json
from pathlib import Path
from PIL import Image

def convert_to_conversation(sample):
    all_objects = (
        sample["categories_in_image"]
    )
    numbered_list = "\n".join(
        f"{i+1}. {obj.replace('_', ' ')}"
        for i, obj in enumerate(all_objects)
    )
    answer = f"The visible entities in the images are:\n{numbered_list}"

    image_path = Path(DATA_DIR) / "/".join(sample['file_path'].split('/')[-2:])
    image = Image.open(image_path)     # .convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text",  "text":  "List everything that is visible in the image."},
                {"type": "image", "image": image}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": answer}
            ]
        },
    ]
    return {"messages": conversation}


To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "user",
  "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
},
{ "role": "assistant",
  "content": [{"type": "text",  "text": A} ]
},
]
```

In [5]:
# instruction = "Write the LaTeX representation for this image."

# def convert_to_conversation(sample):
#     conversation = [
#         { "role": "user",
#           "content" : [
#             {"type" : "text",  "text"  : instruction},
#             {"type" : "image", "image" : sample["image"]} ]
#         },
#         { "role" : "assistant",
#           "content" : [
#             {"type" : "text",  "text"  : sample["text"]} ]
#         },
#     ]
#     return { "messages" : conversation }
# pass

In [6]:
with open(Path(DATA_DIR) / "low/metadata.json", "r") as f_in:
    low_metadata = json.load(f_in)

with open(Path(DATA_DIR) / "medium/metadata.json", "r") as f_in:
    medium_metadata = json.load(f_in)

with open(Path(DATA_DIR) / "high/metadata.json", "r") as f_in:
    high_metadata = json.load(f_in)

Let's convert the dataset into the "correct" format for finetuning:

In [7]:
low_converted_dataset = [convert_to_conversation(sample) for sample in low_metadata]

In [8]:
medium_converted_dataset = [convert_to_conversation(sample) for sample in medium_metadata]

In [9]:
high_converted_dataset = [convert_to_conversation(sample) for sample in high_metadata]

In [10]:
import random
random.seed(3407)

converted_dataset = low_converted_dataset + medium_converted_dataset + high_converted_dataset
random.shuffle(converted_dataset)

We look at how the conversations are structured for the first example:

In [11]:
converted_dataset = converted_dataset[:10000]

In [12]:
print(len(converted_dataset))

10000


In [13]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'List everything that is visible in the image.'},
    {'type': 'image',
     'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': 'The visible entities in the images are:\n1. stop sign'}]}]}

Let's first see before we do any finetuning what the model outputs for the first example!

In [12]:
FastVisionModel.for_inference(model) # Enable for inference!

image = Image.open(Path(DATA_DIR) /  "/".join(medium_metadata[2]["file_path"].split('/')[-2:]))
instruction = "List everything that is visible in the image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True)

The image is a black-and-white photograph of a man in a uniform and a horse standing outside a brick building.

The man is wearing a dark-colored uniform, a helmet, and a long coat. He is standing on the right side of the image, facing the left. The horse is standing in front of him, facing the left side of the image. It has a bridle on its head and is standing in front of an open doorway.

The background of the image is a brick wall with a doorway and a window. The ground is made of dirt or mud.<|eot_id|>


<a name="Train"></a>
### Train the model

In [15]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 30,
        num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = MODEL_SAVE_PATH + "checkpoints/",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [16]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A30. Max memory = 23.598 GB.
9.721 GB of memory reserved.


In [17]:
trainer_stats = trainer.train(resume_from_checkpoint=True)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 1,250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,396,800 of 10,678,617,635 (0.08% trained)


Step,Training Loss
501,0.194500
502,0.098000
503,0.088200
504,0.276400
505,0.243000
506,0.163900
507,0.203500
508,0.230200
509,0.141200
510,0.222100


Unsloth: Will smartly offload gradients to save VRAM!


In [18]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

4836.9262 seconds used for training.
80.62 minutes used for training.
Peak reserved memory = 9.721 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 41.194 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference

In [21]:
FastVisionModel.for_inference(model) # Enable for inference!

image = Image.open(Path(DATA_DIR) / metadata[2]["image_path"])
instruction = "List the objects on the table."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True)

The objects on the table are:
1. bowl
2. bottle
3. whisk
4. spoon
5. clock
6. peanut butter jar
7. can<|im_end|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model.

In [20]:
model.save_pretrained(MODEL_SAVE_PATH)  # Local saving
tokenizer.save_pretrained(MODEL_SAVE_PATH)
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

[]

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [24]:
if True:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = MODEL_SAVE_PATH, # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = True, # Set to False for 16bit LoRA
    )
    FastVisionModel.for_inference(model) # Enable for inference!

image = Image.open(Path(DATA_DIR) / metadata[0]["image_path"])
instruction = "List the objects on the table."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

==((====))==  Unsloth 2026.3.4: Fast Qwen2_5_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
The objects on the table are:
1. whisk
2. clock
3. peanut butter jar
4. can
5. cereal box
6. toothpaste<|im_end|>
